# Light TSMixup

Controlled sinusoidal variant of the Chronos TSMixup augmentation.


This notebook keeps the core mechanism of Chronos Algorithm 1: it samples the number of components, samples the target length, mean-scales each sampled component, draws symmetric Dirichlet weights, and returns a convex combination. The deliberate change for this report is the source pool: Chronos samples subsequences from real training datasets, while Light TSMixup samples from a controlled pool of sinusoidal signals with explicit frequency, amplitude, phase, length, and sampling frequency.


In [ ]:
import numpy as np
from typing import Any, Dict, List, Optional


def _sample_subsequence(series: np.ndarray, length: int, rng: np.random.Generator) -> np.ndarray:
    """Sample a length-`length` subsequence, tiling short sources if needed."""
    series = np.asarray(series, dtype=float)
    T = series.shape[0]
    if T == 0:
        raise ValueError("Base series cannot be empty.")

    if T >= length:
        start = rng.integers(0, T - length + 1)
        return series[start : start + length].copy()

    repeats = (length // T) + 1
    return np.tile(series, repeats)[:length].copy()


def build_sinusoidal_pool(specs: List[Dict[str, float]], fs: float) -> List[Dict[str, Any]]:
    """Create the controlled sinusoidal pool used by Light TSMixup."""
    base_pool = []
    for idx, spec in enumerate(specs, start=1):
        length = int(spec["length"])
        t = np.arange(length) / fs
        values = spec["amplitude"] * np.sin(2 * np.pi * spec["freq_hz"] * t + spec.get("phase", 0.0))
        base_pool.append({
            "name": f"sinusoid_{idx}",
            "freq_hz": float(spec["freq_hz"]),
            "amplitude": float(spec["amplitude"]),
            "phase": float(spec.get("phase", 0.0)),
            "fs": float(fs),
            "length": length,
            "values": values,
        })
    return base_pool


def light_tsmixup(
    base_pool: List[Dict[str, Any]],
    K: int = 3,
    alpha: float = 1.5,
    l_min: int = 128,
    l_max: int = 2048,
    seed: Optional[int] = None,
) -> tuple[np.ndarray, Dict[str, Any]]:
    """Chronos-style TSMixup over a controlled sinusoidal pool."""
    if not base_pool:
        raise ValueError("base_pool must contain at least one source signal.")
    if K < 1:
        raise ValueError("K must be at least 1.")
    if l_min > l_max:
        raise ValueError("l_min must be <= l_max.")

    rng = np.random.default_rng(seed)

    # Algorithm 1, step 1: number of time series to mix.
    k = int(rng.integers(1, K + 1))

    # Algorithm 1, step 2: length of the augmented time series.
    length = int(rng.integers(l_min, l_max + 1))

    scaled_series = []
    sampled_components = []

    for _ in range(k):
        # Light variant: sample from the sinusoidal pool instead of a real dataset collection.
        n = int(rng.integers(0, len(base_pool)))
        component = base_pool[n]
        x = _sample_subsequence(component["values"], length, rng)

        # Chronos TSMixup mean scaling before mixing.
        scale = max(float(np.mean(np.abs(x))), 1e-8)
        scaled_series.append(x / scale)
        sampled_components.append({
            "name": component["name"],
            "freq_hz": component["freq_hz"],
            "amplitude": component["amplitude"],
            "phase": component["phase"],
            "source_length": component["length"],
        })

    # Algorithm 1, step 8: symmetric Dirichlet weights.
    lambdas = rng.dirichlet([alpha] * k)

    # Algorithm 1, step 9: convex combination of mean-scaled series.
    signal = np.sum([weight * series for weight, series in zip(lambdas, scaled_series)], axis=0)

    metadata = {
        "method": "Light TSMixup",
        "k": k,
        "length": length,
        "K": K,
        "alpha": alpha,
        "l_min": l_min,
        "l_max": l_max,
        "lambdas": lambdas,
        "components": sampled_components,
    }
    return signal, metadata


## Controlled sinusoidal pool


In [ ]:
FS = 256.0
SEED = 3

sinusoid_specs = [
    {"freq_hz": 32.0, "amplitude": 0.03, "phase": 0.0, "length": 512},
    {"freq_hz": 64.0, "amplitude": 0.02, "phase": 0.0, "length": 600},
    {"freq_hz": 96.0, "amplitude": 0.02, "phase": np.pi / 4, "length": 700},
    {"freq_hz": 120.0, "amplitude": 0.01, "phase": np.pi / 2, "length": 550},
    {"freq_hz": 128.0, "amplitude": 0.01, "phase": 0.0, "length": 640},
]

base_pool = build_sinusoidal_pool(sinusoid_specs, fs=FS)
print(f"Pool size: {len(base_pool)} controlled sinusoidal signals")


## Light TSMixup signal generation


In [ ]:
augmented, metadata = light_tsmixup(
    base_pool,
    K=3,
    alpha=1.5,
    l_min=512,
    l_max=512,
    seed=SEED,
)

print(f"Shape output: {augmented.shape}")
print(f"Mean: {augmented.mean():.4f}, Std: {augmented.std():.4f}")
print(f"Sampled k: {metadata['k']}, length: {metadata['length']}")
print("Dirichlet weights:", np.round(metadata["lambdas"], 4))
print("Sampled components:")
for component in metadata["components"]:
    print(component)


## Signal plot


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt


def resolve_output_dir() -> Path:
    cwd = Path.cwd()
    for path in [cwd, *cwd.parents]:
        repo_target = path / "chronos" / "data" / "synthetic" / "signals"
        if repo_target.parent.exists():
            return repo_target
        if path.name == "generators" and path.parent.name == "synthetic":
            return path.parent / "signals"
        if path.name == "synthetic" and (path / "generators").exists():
            return path / "signals"
    return Path("chronos/data/synthetic/signals")


output_dir = resolve_output_dir()
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "light_tsmixup.png"

fig, axes = plt.subplots(len(base_pool) + 1, 1, figsize=(12, 3 * (len(base_pool) + 1)))

for i, component in enumerate(base_pool):
    axes[i].plot(component["values"], linewidth=0.8, color=f"C{i}")
    axes[i].set_title(
        f"{component['name']} - {component['freq_hz']} Hz, "
        f"amp={component['amplitude']}, phase={component['phase']:.2f}"
    )
    axes[i].set_ylabel("Value")

axes[-1].plot(augmented, linewidth=0.8, color="crimson")
axes[-1].set_title("Light TSMixup output")
axes[-1].set_ylabel("Value")
axes[-1].set_xlabel("Timestep")

plt.tight_layout()
plt.savefig(output_path, dpi=150)
plt.close(fig)
print(f"Saved in {output_path}")


# KernelSynth


In [ ]:
# KernelSynth implementation aligned with AWS Chronos KernelSynth behavior.
# Keeps explicit matrix construction for didactic/debug purposes.

import numpy as np
import matplotlib.pyplot as plt
from typing import Callable


# ---------------------------------------------------------------------------
# Base kernels
# ---------------------------------------------------------------------------

def kernel_constant(t, tp, C=1.0):
    return np.full((len(t), len(tp)), C, dtype=float)


def kernel_white_noise(t, tp, noise_level=1.0):
    # Same meaning as sklearn WhiteKernel(noise_level=...): value on the diagonal.
    T, Tp = np.meshgrid(t, tp, indexing="ij")
    return noise_level * (T == Tp).astype(float)


def kernel_linear(t, tp, sigma_0=1.0):
    # Equivalent to sklearn DotProduct(sigma_0=sigma_0): sigma_0^2 + x*x'.
    T, Tp = np.meshgrid(t, tp, indexing="ij")
    return sigma_0**2 + T * Tp


def kernel_rbf(t, tp, length_scale=1.0):
    # Equivalent to sklearn RBF(length_scale=...).
    T, Tp = np.meshgrid(t, tp, indexing="ij")
    return np.exp(-0.5 * ((T - Tp) / length_scale) ** 2)


def kernel_rational_quadratic(t, tp, alpha=1.0, length_scale=1.0):
    # Equivalent to sklearn RationalQuadratic(alpha=..., length_scale=1.0).
    T, Tp = np.meshgrid(t, tp, indexing="ij")
    return (1.0 + (T - Tp) ** 2 / (2.0 * alpha * length_scale**2)) ** (-alpha)


def kernel_periodic(t, tp, periodicity=1.0, length_scale=1.0):
    # Equivalent to sklearn ExpSineSquared(periodicity=..., length_scale=1.0).
    T, Tp = np.meshgrid(t, tp, indexing="ij")
    return np.exp(
        -2.0 * np.sin(np.pi * np.abs(T - Tp) / periodicity) ** 2 / length_scale**2
    )


# ---------------------------------------------------------------------------
# AWS-aligned kernel bank
# ---------------------------------------------------------------------------

# Same periodicities as AWS, before normalization by sequence length.
# Duplicates are intentional: AWS has repeated entries, therefore repeated
# kernels must keep higher sampling probability.
AWS_PERIODS = [
    24, 48, 96,
    24 * 7, 48 * 7, 96 * 7,
    7, 14, 30, 60, 365, 365 * 2,
    4, 26, 52,
    4, 6, 12,
    4, 4 * 10,
    10,
]


def build_kernel_bank(l_syn: int = 1024) -> list[tuple[str, Callable]]:
    """Build the KernelSynth bank with AWS-equivalent parameters.

    AWS uses X = linspace(0, 1, LENGTH) and periodicity = raw_period / LENGTH.
    Therefore, if l_syn changes, periodicities must be normalized by l_syn.
    """
    periodic_kernels = [
        (
            f"ExpSineSquared(periodicity={p}/{l_syn})",
            (lambda p=p: lambda t, tp: kernel_periodic(t, tp, periodicity=p / l_syn))(),
        )
        for p in AWS_PERIODS
    ]

    return [
        *periodic_kernels,
        ("DotProduct(sigma_0=0.0)", lambda t, tp: kernel_linear(t, tp, sigma_0=0.0)),
        ("DotProduct(sigma_0=1.0)", lambda t, tp: kernel_linear(t, tp, sigma_0=1.0)),
        ("DotProduct(sigma_0=10.0)", lambda t, tp: kernel_linear(t, tp, sigma_0=10.0)),
        ("RBF(length_scale=0.1)", lambda t, tp: kernel_rbf(t, tp, length_scale=0.1)),
        ("RBF(length_scale=1.0)", lambda t, tp: kernel_rbf(t, tp, length_scale=1.0)),
        ("RBF(length_scale=10.0)", lambda t, tp: kernel_rbf(t, tp, length_scale=10.0)),
        ("RationalQuadratic(alpha=0.1)", lambda t, tp: kernel_rational_quadratic(t, tp, alpha=0.1)),
        ("RationalQuadratic(alpha=1.0)", lambda t, tp: kernel_rational_quadratic(t, tp, alpha=1.0)),
        ("RationalQuadratic(alpha=10.0)", lambda t, tp: kernel_rational_quadratic(t, tp, alpha=10.0)),
        ("WhiteKernel(noise_level=0.1)", lambda t, tp: kernel_white_noise(t, tp, noise_level=0.1)),
        ("WhiteKernel(noise_level=1.0)", lambda t, tp: kernel_white_noise(t, tp, noise_level=1.0)),
        ("ConstantKernel()", lambda t, tp: kernel_constant(t, tp, C=1.0)),
    ]


# ---------------------------------------------------------------------------
# KernelSynth
# ---------------------------------------------------------------------------

def kernel_synth(
    J: int = 5,
    l_syn: int = 1024,
    jitter: float = 0.0,
    return_aws_record: bool = False,
):
    """Generate one KernelSynth time series aligned with AWS behavior.

    Args:
        J: maximum number of kernels to combine.
        l_syn: generated time-series length.
        jitter: optional diagonal stabilization. AWS does not add explicit jitter,
            so the default is 0.0. Use e.g. 1e-6 only if numerical issues appear.
        return_aws_record: if True, return the GluonTS-style dict used by AWS.

    Returns:
        If return_aws_record=False:
            x, kernel_names
        If return_aws_record=True:
            {"start": np.datetime64(...), "target": x}, kernel_names
    """
    kernel_bank = build_kernel_bank(l_syn)

    while True:  # AWS retries when GP sampling fails because of numerical errors.
        t = np.linspace(0, 1, l_syn)

        # Step 1: sample the number of kernels, uniformly in [1, J].
        j = np.random.randint(1, J + 1)

        # Step 2: sample j kernels i.i.d. from the AWS-aligned kernel bank.
        indices = np.random.choice(len(kernel_bank), size=j, replace=True)
        chosen = [kernel_bank[i] for i in indices]

        # Step 3: initialize composite covariance matrix.
        K_star = chosen[0][1](t, t)
        kernel_names = [chosen[0][0]]

        # Step 4: combine kernels using random binary operators (+ or *).
        for name, kfn in chosen[1:]:
            op = np.random.choice(["+", "*"])
            K_new = kfn(t, t)
            if op == "+":
                K_star = K_star + K_new
            else:
                K_star = K_star * K_new
            kernel_names.append(f"{op} {name}")

        if jitter > 0.0:
            K_star = K_star + jitter * np.eye(l_syn)

        # Step 5: sample from GP prior, x ~ N(0, K*).
        # np.random.multivariate_normal accepts positive semidefinite covariance,
        # so it is closer to sklearn's GP sampling than a strict Cholesky-only path.
        try:
            x = np.random.multivariate_normal(mean=np.zeros(l_syn), cov=K_star)
        except np.linalg.LinAlgError as err:
            print("Error caught:", err)
            continue

        if return_aws_record:
            return {
                "start": np.datetime64("2000-01-01 00:00", "s"),
                "target": x,
            }, kernel_names

        return x, kernel_names


# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------

def _clean_kernel_name(name: str) -> str:
    if name.startswith("+ ") or name.startswith("* "):
        return name[2:]
    return name


def plot_kernelsynth(x: np.ndarray, kernel_names: list[str], filepath="kernelsynth.png"):
    t = np.linspace(0, 1, len(x))
    n_kernels = len(kernel_names)
    kernel_bank = build_kernel_bank(len(x))

    fig, axes = plt.subplots(n_kernels + 1, 1, figsize=(12, 3 * (n_kernels + 1)))

    t_small = np.linspace(0, 1, 200)
    for i, name in enumerate(kernel_names):
        clean_name = _clean_kernel_name(name)
        match = next((kfn for kname, kfn in kernel_bank if kname == clean_name), None)
        if match is not None:
            K = match(t_small, t_small)
            axes[i].imshow(K, cmap="viridis", aspect="auto")
            axes[i].set_title(f"Kernel {i + 1}: {name}")
            axes[i].set_xlabel("t")
            axes[i].set_ylabel("t'")
        else:
            axes[i].axis("off")

    axes[-1].plot(t, x, linewidth=0.8, color="crimson")
    axes[-1].set_title("Synthetic Series — KernelSynth AWS-aligned GP sample")
    axes[-1].set_xlabel("Normalized timestep")
    axes[-1].set_ylabel("Value")

    kernel_str = "  |  ".join(kernel_names)
    fig.suptitle(f"Composite kernel: {kernel_str}", fontsize=9, y=1.01)

    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"Saved to '{filepath}'")
    print(f"Kernels used: {kernel_names}")


if __name__ == "__main__":
    x, names = kernel_synth(J=5, l_syn=1024)
    plot_kernelsynth(x, names)
